# CRS correlator demo: corr8, corr32, corr64

This notebook shows the basic use of corriscope. Set `MODE` in the first code cell. The other
cells are the same for the three modes. The notebook does these steps:

1. Program the boards and initialize the array.
2. Get raw ADC samples and plot them.
3. Get raw FFT frames and plot the spectra.
4. Start the correlator and plot the visibilities.

`crs_corr_walkthrough.ipynb` does the same steps. It also computes per-bin digital gains and
compares the ADC model, the firmware FFT and the correlator output.

Before you start:

- Check the hardware map in `SETUP` for your boards.
- Make sure that the capture interface has jumbo frames. Run
  `python -m corriscope.network_utils --status`.
- Make sure that the bitstreams in `corriscope/fpga_firmware/bitstreams/` are about 33 MB each.
  If they are small, run `git lfs pull`.

In [ ]:
import logging, time
import numpy as np
import matplotlib.pyplot as plt
%matplotlib ipympl
plt.rcParams['figure.figsize'] = (11, 5)
logging.getLogger().setLevel(logging.INFO)

from corriscope.fpga_array import FPGAArray

MODE = 'corr8'     # 'corr8' | 'corr32' | 'corr64'

# Hardware maps. 'crs *:N' is the board in crate slot N, whatever its serial number.
# With more than one crate on the network, name the boards instead: 'crs 127:1 134:2 ...'.
SYNC_TRIG = dict(sync_method='trig', sync_source='bp_trig',
                 sync_master=1, sync_master_source='sync_out', sync_master_output=1)
SETUP = {
    # one board in slot 1; software sync
    'corr8':  dict(hwm='crs *:1', sync_method='local'),
    # four boards in slots 1-4 (one backplane); the board in slot 1 sends the trigger
    'corr32': dict(hwm='crs *:1 *:2 *:3 *:4', **SYNC_TRIG),
    # eight boards in slots 1-8 (two backplanes with SFP links); the board in slot 1 sends the trigger
    'corr64': dict(hwm='crs *:1 *:2 *:3 *:4 *:5 *:6 *:7 *:8', **SYNC_TRIG),
}

## 1. Program and initialize

`mode` selects the bitstream `chfpga_crs_<mode>.bit` and the initialization sequence. The
boards are programmed each time. To keep a programmed FPGA, add `prog=1`.

In [ ]:
ca = await FPGAArray(mode=MODE, stderr_log_level='info', **SETUP[MODE])

fs = ca.ib[0]._sampling_frequency          # ADC sampling frequency, Hz (3.2 GHz)
frame_len = ca.ib[0].FRAME_LENGTH          # ADC samples per frame (16384)
nbins = ca.ib[0].NUMBER_OF_FREQUENCY_BINS  # FFT bins per frame (8192)
nchan = ca.ib[0].NUMBER_OF_CHANNELIZERS    # inputs per board (8)
print(f'{len(ca.ib)} board(s): {[str(b) for b in ca.ib]}')
print(f'firmware {await ca.ib[0].get_fpga_firmware_timestamp_async()}, fs={fs/1e6:.0f} MHz, '
      f'{frame_len} samples/frame, {nbins} bins')
print('SYNC counters (all values must be equal):', list(ca.ib.REFCLK.SYNC_CTR))

The receivers return the inputs in the order of `ca.ib`, then the order of the channels on
each board. The flat input index is `board_index * 8 + channel`.

In [ ]:
labels = [f'{b.get_string_id()} ch{ch}' for b in ca.ib for ch in range(nchan)]
print(len(labels), 'inputs:', labels[:4], '...')

## 2. Raw ADC samples

UCAP captures the output of FUNCGEN. With `data_source='adc'`, that output is the ADC
samples. UCAP `mode=0` sends 2 frames of all 8 inputs per burst. The receiver returns 14-bit
signed integers.

In [ ]:
await ca.set_channelizers_async(data_source='adc', funcgen_left_shift=0, funcgen_right_shift=0,
                                fft_bypass=False, scaler_bypass=False,
                                offset_binary_encoding=False, sync=False)
await ca.start_data_capture_async(0.02, source='funcgen', mode=0, data_type=None, sync=False)

r = await ca.get_data_receiver_async(tcp=False)
t_adc, adc, cnt = r.read_raw_frames(ncap=4, split=True, flush=True)   # adc: (inputs, frames, frame_len)
print(f'ADC data {adc.shape} dtype={adc.dtype}  min={adc.min()} max={adc.max()}  '
      f'(the 14-bit range is -8192..8191)')

In [ ]:
inputs_to_plot = [0, 1, 2, 3]       # flat input indices
nsamp = 2000
tt = np.arange(nsamp) / fs * 1e9     # ns

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(14, 4.5))
for i in inputs_to_plot:
    ax0.plot(tt, adc[i, 0, :nsamp], lw=0.8, label=labels[i])
    ax1.hist(adc[i].ravel(), bins=128, histtype='step', label=f'{labels[i]}  rms={adc[i].std():.1f}')
ax0.set(xlabel='time [ns]', ylabel='ADC code (14 bit)', title='raw ADC samples, first frame'); ax0.grid(); ax0.legend(fontsize=8)
ax1.set(xlabel='ADC code', ylabel='count', title='sample histogram'); ax1.set_yscale('log'); ax1.grid(); ax1.legend(fontsize=8)

rms = adc.reshape(adc.shape[0], -1).std(axis=1)
plt.figure(figsize=(12, 3.5))
plt.bar(range(len(rms)), rms)
plt.xticks(range(len(rms)), labels, rotation=90, fontsize=7)
plt.ylabel('rms [ADC codes]'); plt.title('ADC rms per input'); plt.grid(axis='y')

## 3. Raw FFT frames

`data_type='fft_2x'` captures the FFT output before the gain, with 32+32 bits. The packet
processor puts the bins in frequency order. UCAP `mode=0` gives one spectrum for all 8 inputs
of each board per burst.

Verify on hardware: the reference test captured `fft_2x` with `mode=3, channels=[0]`. If
this cell fails, use those arguments.

In [ ]:
await ca.start_data_capture_async(0.05, source='scaler', mode=0, data_type='fft_2x', sync=False)
t_fft, fft, cnt = r.read_raw_frames(ncap=16, split=True, flush=True)   # fft: (inputs, frames, nbins) complex
print(f'FFT data {fft.shape} {fft.dtype}')

freq = np.arange(fft.shape[-1]) * fs / 2 / fft.shape[-1] / 1e6   # MHz, first Nyquist zone
fft_pow = (np.abs(fft) ** 2).mean(axis=1)                        # mean |X|^2 per input and bin

plt.figure(figsize=(12, 5))
for i in inputs_to_plot:
    plt.semilogy(freq[1:], fft_pow[i, 1:], lw=0.7, label=labels[i])
plt.xlabel('frequency [MHz]'); plt.ylabel('|FFT|$^2$ (32+32-bit units)'); plt.title('firmware FFT, mean power spectrum')
plt.grid(); plt.legend(fontsize=8)

## 4. Correlator and visibilities

The scaler multiplies each bin by a digital gain and keeps 4+4 bits. This cell sets one flat
gain for all bins. The walkthrough notebook computes a gain for each bin from the FFT data.

The firmware accumulates `integ_period` frames. Each board correlates a subset of the bins,
and the subset changes after each integration. `soft_integ_period=8` sums 8 integrations in
software, so that the result has all bins. `cr.get_good_bin_mask()` marks the bins that have
data. `cr.get_ij_to_prod_map()[i, j]` is the product index of inputs `i` and `j`.

In [ ]:
await ca.set_channelizers_async(gains=1 << 25, sync=False)   # flat digital gain

integ_period = 32768                 # frames per firmware integration (8192..32768)
soft_integ_period = 8                # firmware integrations summed in software
await ca.start_correlators_async(integ_period)

cr = await ca.get_corr_receiver_async()
ijmap = cr.get_ij_to_prod_map()
good = cr.get_good_bin_mask()
print(f'correlator: {ijmap.shape[0]} inputs, {ijmap.max()+1} products, {good.sum()} of {good.size} bins have data')

for b in ca.ib:                      # clear the overrun flags before the read
    b.CORR.OVERRUN_RESET = 1; b.CORR.OVERRUN_RESET = 0
    b.UCAP.USER_RESET = 1;    b.UCAP.USER_RESET = 0

t_vis, vis, vis_cnt, vis_sat = cr.read_corr_frames(flush=True, soft_integ_period=soft_integ_period,
                                                   number_of_results=2, verbose=1)
print(f'visibilities {vis.shape} (results, bins, products), counts={vis_cnt}, saturations={int(vis_sat.sum())}')
print('overruns  UCAP:', list(ca.ib.UCAP.OVERRUN), ' CORR:', list(ca.ib.CORR.OVERRUN), '(all values must be 0)')

In [ ]:
# autocorrelations of some inputs, and one cross-correlation (amplitude and phase)
i0, j0 = inputs_to_plot[0], inputs_to_plot[1]
v = vis[0]
fig, (ax0, ax1) = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
for i in inputs_to_plot:
    ax0.semilogy(freq[good], np.abs(v[good, ijmap[i, i]]), '.', ms=2, label=f'auto {labels[i]}')
ax0.semilogy(freq[good], np.abs(v[good, ijmap[i0, j0]]), '.', ms=2, label=f'cross ({i0},{j0})')
ax0.set_ylabel('|V|'); ax0.grid(); ax0.legend(fontsize=8); ax0.set_title('visibilities, first result')
ax1.plot(freq[good], np.degrees(np.angle(v[good, ijmap[i0, j0]])), '.', ms=2)
ax1.set(xlabel='frequency [MHz]', ylabel='phase [deg]', title=f'cross-correlation phase ({labels[i0]}) x ({labels[j0]})'); ax1.grid()

## 5. Stop

This cell stops the data flow. The boards stay programmed. In `corr32` and `corr64`, it also
prints the corner-turn alignment errors. All values must be 0.

In [ ]:
if MODE != 'corr8':
    for b in ca.ib:
        for ct in (b.CT.CT2, b.CT.CT3):
            ct.ALIGN_MON_RESET = 1; ct.ALIGN_MON_RESET = 0; ct.ALIGN_MON_SOURCE = 0
    time.sleep(0.3)
    print('CT2 align errors:', [int(b.CT.CT2.ALIGN_MON_WORD) for b in ca.ib])
    print('CT3 align errors:', [int(b.CT.CT3.ALIGN_MON_WORD) for b in ca.ib])

for b in ca.ib:
    b.stop_data_capture()
    b.CORR.stop_correlator()